# Use Lasso to predict treatment (binary), then see which genes are most important to predict

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("LASSO_bootstrap").getOrCreate()
sc = spark.sparkContext
print(sc.defaultParallelism)

2


26/05/30 00:58:20 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
import pandas as pd
import numpy as np
import pyspark
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LASSO_bootstrap").getOrCreate()
sc = spark.sparkContext

cpm = pd.read_parquet('gs://gene_datasets/sample_by_gene.parquet')
pca_summary = pd.read_csv('gs://gene_datasets/pca_summary.csv')

pareto_treatments = pca_summary.loc[pca_summary['pareto_optimal'], 'sample_type'].tolist()
print(f"Pareto-optimal treatments ({len(pareto_treatments)}): {pareto_treatments}")

Pareto-optimal treatments (5): ['Base', 'hATF567', 'hATF561', 'nZF105', 'nZF139']


In [4]:
# Filter to pareto treatments + Control
mask = cpm['sample_type'].isin(pareto_treatments + ['Control'])
sub = cpm[mask].copy()
gene_cols = [c for c in sub.columns if c.startswith('ENSG')]

print(f"Samples: {len(sub)}, Genes: {len(gene_cols)}")
print(sub['sample_type'].value_counts())

X_scaled = StandardScaler().fit_transform(sub[gene_cols].values)
y = (sub['sample_type'] != 'Control').astype(int).values

assert (y == 0).any() and (y == 1).any(), "y must contain both classes before bootstrapping"

X_broadcast = sc.broadcast(X_scaled)
y_broadcast = sc.broadcast(y)

def fit_one_bootstrap(seed):
    import numpy as np
    from sklearn.linear_model import LogisticRegression
    
    X, y_local = X_broadcast.value, y_broadcast.value
    rng = np.random.RandomState(seed)
    
    treat_idx = np.where(y_local == 1)[0]
    control_idx = np.where(y_local == 0)[0]

    if len(treat_idx) == 0 or len(control_idx) == 0:
        return [0] * X.shape[1]
    
    boot_treat = rng.choice(treat_idx, len(treat_idx), replace=True)
    boot_control = rng.choice(control_idx, len(control_idx), replace=True)
    
    idx = np.concatenate([boot_treat, boot_control])
    y_boot = y_local[idx]

    if len(np.unique(y_boot)) < 2:
        return [0] * X.shape[1]
    
    lasso = LogisticRegression(
        penalty='l1', solver='liblinear', C=10.0, max_iter=5000
    )
    lasso.fit(X[idx], y_boot)
    return (lasso.coef_[0] != 0).astype(int).tolist()

N_BOOTSTRAP = 50
seeds_rdd = sc.parallelize(range(N_BOOTSTRAP), numSlices=10)
selection_results = seeds_rdd.map(fit_one_bootstrap).collect()

selection_freq = np.array(selection_results).sum(axis=0) / N_BOOTSTRAP

# Show distribution of selection frequencies before filtering
freq_series = pd.Series(selection_freq, index=gene_cols).sort_values(ascending=False)
print("\nSelection frequency distribution (genes selected in >0% of bootstraps):")
print(freq_series[freq_series > 0].describe())
print(f"\nTop 10 genes by selection frequency:")
print(freq_series.head(10))

stable_genes = pd.DataFrame({
    'gene_id': gene_cols,
    'selection_frequency': selection_freq,
}).query('selection_frequency >= 0.5').sort_values('selection_frequency', ascending=False)

print(f"\nStably selected genes (>=50% of bootstraps): {len(stable_genes)}")
print(stable_genes.head(30))

Samples: 12, Genes: 78893
Base        2
hATF561     2
hATF567     2
nZF105      2
nZF139      2
Control     2
nZF153      0
nZF93       0
nZF81       0
nZF42       0
nZF36       0
nZF156      0
nZF154      0
nZF147      0
nZF151      0
nZF148      0
nZF145      0
hATF555R    0
hATF555Q    0
ZDS2        0
SP1R        0
nZFD96      0
Name: sample_type, dtype: int64



Selection frequency distribution (genes selected in >0% of bootstraps):
count    3247.000000
mean        0.024527
std         0.010993
min         0.020000
25%         0.020000
50%         0.020000
75%         0.020000
max         0.120000
dtype: float64

Top 10 genes by selection frequency:
ENSG00000269082    0.12
ENSG00000271998    0.10
ENSG00000138760    0.10
ENSG00000166825    0.10
ENSG00000243686    0.10
ENSG00000232859    0.08
ENSG00000231705    0.08
ENSG00000238171    0.08
ENSG00000198711    0.08
ENSG00000104435    0.08
dtype: float64

Stably selected genes (>=50% of bootstraps): 0
Empty DataFrame
Columns: [gene_id, selection_frequency]
Index: []


We applied bootstrap-stabilized LASSO to identify genes distinguishing Pareto-optimal treatments from control. No gene was selected in more than 6% of bootstraps, indicating that no single gene reliably discriminates treatment status given the available sample size. Instead, treatment effects are diffuse across many weakly-discriminating genes, consistent with the broad transcriptomic perturbation observed in PCA.

# Problem with lasso approach (keep for showing trial and error and pivoting)
essentially the issue here is that Lasso, even with bootstrapping only has 3 options with the 2 controls. We have (C1, C2), (C1, C1), (C2, C2). This is a really small sample space and so we fail to accurately model variance, so the confidence intervals around the Lasso coefficients will be incredibly tight.

Second, only two samples cannot capture the natural variance in the control population. In an 80k dimensional space the probably that some genes will randomly have exteme values in those two specific samples compared to the 22 treatments is kinda high

Lastly, the 22-2 ratio is really imbalanced, and lasso can only handle at most n variables before it gets really saturated so we can at most only select 24/80k. 

# Solution to Lasso problem (maybe?)




In [7]:
# Get unique gene IDs that have any Open Targets association
disease_associations = pd.read_parquet('gs://gene_datasets/disease_associations.parquet')
clinically_relevant_genes = set(disease_associations['gene_id'].unique())
print(f"Genes with Open Targets associations: {len(clinically_relevant_genes)}")

# Filter your gene matrix to just those
filtered_gene_cols = [g for g in gene_cols if g in clinically_relevant_genes]
print(f"Filtered from {len(gene_cols)} to {len(filtered_gene_cols)} genes")

# Build filtered X, scale, broadcast
X_filtered = sub[filtered_gene_cols].values
X_filtered_scaled = StandardScaler().fit_transform(X_filtered)

X_broadcast = sc.broadcast(X_filtered_scaled)  # overwrite previous broadcast
# y_broadcast is unchanged

# Update gene_cols to the filtered list so downstream uses the right names
gene_cols = filtered_gene_cols

Genes with Open Targets associations: 18845
Filtered from 78893 to 18845 genes


In [11]:
def fit_one_nsc_bootstrap(seed):
    import numpy as np
    from sklearn.neighbors import NearestCentroid
    
    X, y_local = X_broadcast.value, y_broadcast.value
    rng = np.random.RandomState(seed)
    
    treat_idx = np.where(y_local == 1)[0]
    control_idx = np.where(y_local == 0)[0]
    idx = np.concatenate([
        rng.choice(treat_idx, len(treat_idx), replace=True),
        rng.choice(control_idx, len(control_idx), replace=True),
    ])
    
    nsc = NearestCentroid(shrink_threshold=THRESHOLD_VAL)
    nsc.fit(X[idx], y_local[idx])
    
    # Genes with non-zero centroid difference between classes are "selected"
    diff = nsc.centroids_[1] - nsc.centroids_[0]
    return (diff != 0).astype(int).tolist()

N_BOOTSTRAP = 50
seeds_rdd = sc.parallelize(range(N_BOOTSTRAP), numSlices=10)

THRESHOLD = [0.5, 1.0, 1.5, 2.0] # test some different levels

for THRESHOLD_VAL in THRESHOLD:
    print("Testing value:", THRESHOLD_VAL)
    selection_results = seeds_rdd.map(fit_one_nsc_bootstrap).collect()

    selection_freq = np.array(selection_results).sum(axis=0) / N_BOOTSTRAP

    nsc_stability = pd.DataFrame({
        'gene_id': gene_cols,
        'selection_frequency': selection_freq,
    })

    print("Selection frequency distribution:")
    print(nsc_stability['selection_frequency'].describe())
    print(f"\nGenes selected in >=50% of bootstraps: {(nsc_stability['selection_frequency'] >= 0.5).sum()}")
    print(f"Genes selected in >=70% of bootstraps: {(nsc_stability['selection_frequency'] >= 0.7).sum()}")
    print(f"Genes selected in 100% of bootstraps: {(nsc_stability['selection_frequency'] == 1.0).sum()}")

    print("\nTop 20 stable genes:")
    print(nsc_stability.nlargest(20, 'selection_frequency'))

Testing value: 0.5


Selection frequency distribution:
count    18845.000000
mean         0.459271
std          0.312073
min          0.000000
25%          0.180000
50%          0.460000
75%          0.700000
max          1.000000
Name: selection_frequency, dtype: float64

Genes selected in >=50% of bootstraps: 8957
Genes selected in >=70% of bootstraps: 4808
Genes selected in 100% of bootstraps: 1227

Top 20 stable genes:
             gene_id  selection_frequency
15   ENSG00000001629                  1.0
17   ENSG00000001631                  1.0
31   ENSG00000003096                  1.0
37   ENSG00000003402                  1.0
47   ENSG00000004455                  1.0
65   ENSG00000004897                  1.0
72   ENSG00000005020                  1.0
95   ENSG00000005471                  1.0
96   ENSG00000005483                  1.0
99   ENSG00000005700                  1.0
101  ENSG00000005810                  1.0
141  ENSG00000006634                  1.0
156  ENSG00000006837                  1.0
166  E

Selection frequency distribution:
count    18845.000000
mean         0.196291
std          0.244131
min          0.000000
25%          0.000000
50%          0.080000
75%          0.320000
max          1.000000
Name: selection_frequency, dtype: float64

Genes selected in >=50% of bootstraps: 2343
Genes selected in >=70% of bootstraps: 1176
Genes selected in 100% of bootstraps: 210

Top 20 stable genes:
              gene_id  selection_frequency
99    ENSG00000005700                  1.0
166   ENSG00000007202                  1.0
199   ENSG00000008196                  1.0
206   ENSG00000008294                  1.0
316   ENSG00000012983                  1.0
433   ENSG00000025039                  1.0
487   ENSG00000033030                  1.0
515   ENSG00000035928                  1.0
613   ENSG00000047597                  1.0
638   ENSG00000048828                  1.0
739   ENSG00000055732                  1.0
777   ENSG00000058668                  1.0
861   ENSG00000064218               

Selection frequency distribution:
count    18845.000000
mean         0.074527
std          0.139986
min          0.000000
25%          0.000000
50%          0.000000
75%          0.080000
max          1.000000
Name: selection_frequency, dtype: float64

Genes selected in >=50% of bootstraps: 382
Genes selected in >=70% of bootstraps: 158
Genes selected in 100% of bootstraps: 33

Top 20 stable genes:
              gene_id  selection_frequency
99    ENSG00000005700                  1.0
613   ENSG00000047597                  1.0
739   ENSG00000055732                  1.0
861   ENSG00000064218                  1.0
1027  ENSG00000068489                  1.0
1095  ENSG00000070476                  1.0
3479  ENSG00000108588                  1.0
3609  ENSG00000109689                  1.0
4169  ENSG00000114248                  1.0
5040  ENSG00000121316                  1.0
5626  ENSG00000126070                  1.0
5896  ENSG00000128829                  1.0
5968  ENSG00000129480                  

Selection frequency distribution:
count    18845.000000
mean         0.028322
std          0.078040
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: selection_frequency, dtype: float64

Genes selected in >=50% of bootstraps: 46
Genes selected in >=70% of bootstraps: 21
Genes selected in 100% of bootstraps: 11

Top 20 stable genes:
               gene_id  selection_frequency
613    ENSG00000047597                 1.00
739    ENSG00000055732                 1.00
861    ENSG00000064218                 1.00
6382   ENSG00000132446                 1.00
6478   ENSG00000132958                 1.00
6593   ENSG00000133958                 1.00
6716   ENSG00000134627                 1.00
7904   ENSG00000141622                 1.00
13718  ENSG00000178295                 1.00
15515  ENSG00000188817                 1.00
15617  ENSG00000189350                 1.00
11855  ENSG00000168135                 0.98
17409  ENSG00000226174      

tuned threshold in NSC without cv because using any sort of CV might just not have controls in there, and 11 fold CV doesnt work becuase we will have a fold with no controls so we just need to experiment

In [15]:
# Updated bootstrap function that takes (seed, threshold) tuple
def fit_one_nsc_bootstrap_with_threshold(seed_and_threshold):
    import numpy as np
    from sklearn.neighbors import NearestCentroid
    
    seed, threshold = seed_and_threshold
    X, y_local = X_broadcast.value, y_broadcast.value
    rng = np.random.RandomState(seed)
    treat_idx = np.where(y_local == 1)[0]
    control_idx = np.where(y_local == 0)[0]
    idx = np.concatenate([
        rng.choice(treat_idx, len(treat_idx), replace=True),
        rng.choice(control_idx, len(control_idx), replace=True),
    ])
    
    nsc = NearestCentroid(shrink_threshold=threshold)
    nsc.fit(X[idx], y_local[idx])
    diff = nsc.centroids_[1] - nsc.centroids_[0]
    return (threshold, (diff != 0).astype(int).tolist())

THRESHOLDS = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
N_BOOTSTRAP = 50

# Build all (seed, threshold) jobs and run in parallel in one shot
all_jobs = [(seed, t) for t in THRESHOLDS for seed in range(N_BOOTSTRAP)]
jobs_rdd = sc.parallelize(all_jobs, numSlices=20)
all_results = jobs_rdd.map(fit_one_nsc_bootstrap_with_threshold).collect()

# Aggregate per threshold, save stable gene sets for intersection
threshold_stable_sets = {}
threshold_stability_dfs = {}

for threshold in THRESHOLDS:
    selections = [r[1] for r in all_results if r[0] == threshold]
    freq = np.array(selections).sum(axis=0) / N_BOOTSTRAP
    
    nsc_stability = pd.DataFrame({
        'gene_id': gene_cols,
        'selection_frequency': freq,
    })
    threshold_stability_dfs[threshold] = nsc_stability
    
    # Save the 100%-stable gene set for the intersection check
    threshold_stable_sets[threshold] = set(nsc_stability.query('selection_frequency == 1.0')['gene_id'])
    
    n_at_100 = (freq == 1.0).sum()
    n_at_70 = (freq >= 0.7).sum()
    n_at_50 = (freq >= 0.5).sum()
    print(f"Threshold {threshold}: {n_at_100} at 100%, {n_at_70} at >=70%, {n_at_50} at >=50%")

# Robustness check: genes 100%-stable across ALL thresholds tested
robust_genes = set.intersection(*threshold_stable_sets.values())
print(f"\nGenes 100%-stable across ALL tested thresholds: {len(robust_genes)}")

# Also check the more practical intersection: stable at strict thresholds only
# (drop threshold 0.5 since it's typically too permissive to be meaningful)
strict_robust = set.intersection(*[threshold_stable_sets[t] for t in THRESHOLDS if t >= 1.0])
print(f"Genes 100%-stable across thresholds >=1.0: {len(strict_robust)}")

print(f"\nRobust gene IDs (stable across thresholds >=1.0):")
print(sorted(strict_robust))

Threshold 0.5: 1227 at 100%, 4808 at >=70%, 8957 at >=50%
Threshold 1.0: 210 at 100%, 1176 at >=70%, 2343 at >=50%
Threshold 1.5: 33 at 100%, 158 at >=70%, 382 at >=50%
Threshold 2.0: 11 at 100%, 21 at >=70%, 46 at >=50%
Threshold 2.5: 4 at 100%, 6 at >=70%, 10 at >=50%
Threshold 3.0: 3 at 100%, 4 at >=70%, 4 at >=50%

Genes 100%-stable across ALL tested thresholds: 3
Genes 100%-stable across thresholds >=1.0: 3

Robust gene IDs (stable across thresholds >=1.0):
['ENSG00000047597', 'ENSG00000064218', 'ENSG00000132446']


26/05/30 01:22:25 WARN BlockManagerMasterEndpoint: No more replicas available for broadcast_5_python !
26/05/30 01:22:25 WARN BlockManagerMasterEndpoint: No more replicas available for broadcast_2_python !


In [ ]:
def fit_one_nsc_permuted(args):
    import numpy as np
    from sklearn.neighbors import NearestCentroid
    
    seed, threshold = args
    X = X_broadcast.value
    y_true = y_broadcast.value
    
    rng = np.random.RandomState(seed)
    y_permuted = rng.permutation(y_true)
    
    nsc = NearestCentroid(shrink_threshold=threshold)
    nsc.fit(X, y_permuted)
    diff = nsc.centroids_[1] - nsc.centroids_[0]
    return (threshold, (diff != 0).sum())  # just count, don't track which genes

N_PERMUTATIONS = 100
permutation_jobs = [(seed, t) for t in THRESHOLDS for seed in range(N_PERMUTATIONS)]
perm_rdd = sc.parallelize(permutation_jobs, numSlices=20)
perm_results = perm_rdd.map(fit_one_nsc_permuted).collect()

print(f"\n{'Threshold':<10} {'Real (any stability)':<22} {'Random mean ± std':<22} {'P-value approx':<15}")
for threshold in THRESHOLDS:
    perm_counts = [r[1] for r in perm_results if r[0] == threshold]
    # Your real selection counts at this threshold (use 50% stability for "selected")
    real_count = (threshold_stability_dfs[threshold]['selection_frequency'] >= 0.5).sum()
    perm_mean = np.mean(perm_counts)
    perm_std = np.std(perm_counts)
    # Approx p-value: fraction of permutations that got at least as many genes
    p = np.mean([c >= real_count for c in perm_counts])
    print(f"{threshold:<10} {real_count:<22} {perm_mean:.0f} ± {perm_std:.0f}{'':<10} {p:.3f}")


Threshold  Real (any stability)   Random mean ± std      P-value approx 
0.5        8957                   5554 ± 864           0.000
1.0        2343                   1014 ± 353           0.000
1.5        382                    82 ± 45           0.000
2.0        46                     10 ± 4           0.000
2.5        10                     5 ± 3           0.050
3.0        4                      5 ± 3           0.670


26/05/30 01:26:43 WARN BlockManagerMasterEndpoint: No more replicas available for broadcast_5_python !
26/05/30 01:26:43 WARN BlockManagerMasterEndpoint: No more replicas available for broadcast_2_python !


In [17]:
REFINE_THRESHOLDS = [2.0, 2.1, 2.2, 2.3, 2.4, 2.5]

N_PERMUTATIONS = 100
permutation_jobs = [(seed, t) for t in REFINE_THRESHOLDS for seed in range(N_PERMUTATIONS)]
perm_rdd = sc.parallelize(permutation_jobs, numSlices=20)
perm_results = perm_rdd.map(fit_one_nsc_permuted).collect()

print(f"\n{'Threshold':<10} {'Real (any stability)':<22} {'Random mean ± std':<22} {'P-value approx':<15}")
for threshold in REFINE_THRESHOLDS:
    perm_counts = [r[1] for r in perm_results if r[0] == threshold]
    # Your real selection counts at this threshold (use 50% stability for "selected")
    real_count = (threshold_stability_dfs[threshold]['selection_frequency'] >= 0.5).sum()
    perm_mean = np.mean(perm_counts)
    perm_std = np.std(perm_counts)
    # Approx p-value: fraction of permutations that got at least as many genes
    p = np.mean([c >= real_count for c in perm_counts])
    print(f"{threshold:<10} {real_count:<22} {perm_mean:.0f} ± {perm_std:.0f}{'':<10} {p:.3f}")


Threshold  Real (any stability)   Random mean ± std      P-value approx 
2.0        46                     10 ± 4           0.000


KeyError: 2.1

26/05/30 01:30:46 WARN BlockManagerMasterEndpoint: No more replicas available for broadcast_5_python !
26/05/30 01:30:46 WARN BlockManagerMasterEndpoint: No more replicas available for broadcast_2_python !


In [18]:
REFINE_THRESHOLDS = [2.0, 2.1, 2.2, 2.3, 2.4, 2.5]
N_BOOTSTRAP = 50
N_PERMUTATIONS = 100

# Step 1 — bootstrap stability at the new thresholds
boot_jobs = [(seed, t) for t in REFINE_THRESHOLDS for seed in range(N_BOOTSTRAP)]
boot_rdd = sc.parallelize(boot_jobs, numSlices=20)
boot_results = boot_rdd.map(fit_one_nsc_bootstrap_with_threshold).collect()

# Compute real selection counts per threshold
real_counts = {}
for threshold in REFINE_THRESHOLDS:
    selections = [r[1] for r in boot_results if r[0] == threshold]
    freq = np.array(selections).sum(axis=0) / N_BOOTSTRAP
    real_counts[threshold] = (freq >= 0.5).sum()  # genes at >=50% stability

# Step 2 — permutation null at the new thresholds
perm_jobs = [(seed, t) for t in REFINE_THRESHOLDS for seed in range(N_PERMUTATIONS)]
perm_rdd = sc.parallelize(perm_jobs, numSlices=20)
perm_results = perm_rdd.map(fit_one_nsc_permuted).collect()

# Step 3 — report
print(f"\n{'Threshold':<12} {'Real (>=50% stable)':<22} {'Random mean ± std':<22} {'P-value':<10}")
for threshold in REFINE_THRESHOLDS:
    perm_counts = [r[1] for r in perm_results if r[0] == threshold]
    perm_mean = np.mean(perm_counts)
    perm_std = np.std(perm_counts)
    p = np.mean([c >= real_counts[threshold] for c in perm_counts])
    print(f"{threshold:<12} {real_counts[threshold]:<22} {perm_mean:.0f} ± {perm_std:.0f}{'':<14} {p:.3f}")


Threshold    Real (>=50% stable)    Random mean ± std      P-value   
2.0          46                     10 ± 4               0.000
2.1          32                     8 ± 4               0.000
2.2          19                     7 ± 3               0.000
2.3          14                     6 ± 3               0.020
2.4          13                     6 ± 3               0.020
2.5          10                     5 ± 3               0.050


We selected threshold 2.2 as the most stringent threshold maintaining strong significance (p < 0.001) against the permutation null. This yielded 19 genes selected in ≥50% of bootstraps. We also report results at threshold 2.0 (46 genes, p < 0.001) as a more inclusive view for sensitivity analysis."

In [ ]:
freq_22 = np.array([r[1] for r in boot_results if r[0] == 2.2]).sum(axis=0) / N_BOOTSTRAP
threshold_stability_dfs[2.2] = pd.DataFrame({
    'gene_id': gene_cols,
    'selection_frequency': freq_22,
})

for threshold in [2.0, 2.2]:
    stable = (threshold_stability_dfs[threshold]
              .query('selection_frequency >= 0.5')
              .sort_values('selection_frequency', ascending=False))
    print(f"\n{'='*60}")
    print(f"THRESHOLD {threshold}: {len(stable)} genes at >=50% stability")
    print(f"{'='*60}")
    print(stable.to_string(index=False))


THRESHOLD 2.0: 46 genes at >=50% stability
        gene_id  selection_frequency
ENSG00000141622                 1.00
ENSG00000132446                 1.00
ENSG00000178295                 1.00
ENSG00000047597                 1.00
ENSG00000188817                 1.00
ENSG00000134627                 1.00
ENSG00000133958                 1.00
ENSG00000132958                 1.00
ENSG00000189350                 1.00
ENSG00000055732                 1.00
ENSG00000064218                 1.00
ENSG00000168135                 0.98
ENSG00000226174                 0.92
ENSG00000114248                 0.86
ENSG00000276368                 0.84
ENSG00000109689                 0.78
ENSG00000108588                 0.72
ENSG00000139985                 0.72
ENSG00000140386                 0.72
ENSG00000068489                 0.70
ENSG00000182934                 0.70
ENSG00000095739                 0.68
ENSG00000214029                 0.64
ENSG00000129480                 0.62
ENSG00000187742                

The actual computation you ran was:

6 thresholds (initial) + 6 thresholds (refinement) = 12 thresholds
50 bootstraps + 100 permutations per threshold = 150 iterations per threshold
12 × 150 = 1,800 independent NSC fits
Each fit is small (NSC is fast), but 1,800 of them is enough that single-threaded execution would take noticeable time and benefit from parallelization. This is the canonical "embarrassingly parallel" workload Spark exists for — many independent jobs with no cross-talk needed between workers.

# Now Enrich these results with opentargets

In [ ]:
# Pick your stable gene set (using threshold 2.2 as primary)
PRIMARY_THRESHOLD = 2.2
stable_genes = (threshold_stability_dfs[PRIMARY_THRESHOLD]
                .query('selection_frequency >= 0.5'))

# Enrich
nsc_enriched = stable_genes.merge(
    disease_associations, on='gene_id', how='left'
)

print(f"Stable genes: {len(stable_genes)}")
print(f"Genes with at least one disease association: "
      f"{nsc_enriched.dropna(subset=['disease_name'])['gene_id'].nunique()}")

# === Top diseases linked to stable genes ===
top_diseases = (nsc_enriched
                .dropna(subset=['disease_name'])
                .groupby('disease_name')
                .agg(n_genes=('gene_id', 'nunique'),
                     avg_score=('association_score', 'mean'),
                     max_score=('association_score', 'max'))
                .query('n_genes >= 2')  # at least 2 stable genes converge on the trait
                .sort_values(['n_genes', 'avg_score'], ascending=False)
                .head(20))

# === Top therapeutic areas ===
exploded = (nsc_enriched
            .dropna(subset=['disease_name'])
            .explode('therapeutic_areas')
            .dropna(subset=['therapeutic_areas']))

top_areas = (exploded
             .groupby('therapeutic_areas')
             .agg(n_genes=('gene_id', 'nunique'),
                  n_diseases=('disease_name', 'nunique'),
                  avg_score=('association_score', 'mean'))
             .sort_values('n_genes', ascending=False))


# === Drill into the 3 ultra-robust genes ===
ultra_robust_ids = ['ENSG00000047597', 'ENSG00000064218', 'ENSG00000132446']
ultra_robust_details = (nsc_enriched[nsc_enriched['gene_id'].isin(ultra_robust_ids)]
                        .dropna(subset=['disease_name'])
                        .sort_values(['gene_id', 'association_score'], ascending=[True, False]))

# Save final results to GCS
#stable_genes.to_csv('gs://gene_datasets/nsc_stable_genes_t22.csv', index=False)
#top_diseases.to_csv('gs://gene_datasets/nsc_top_diseases.csv')
#top_areas.to_csv('gs://gene_datasets/nsc_top_areas.csv')
#print("\nSaved results to gs://gene_datasets/")

Stable genes: 19
Genes with at least one disease association: 19

Saved results to gs://gene_datasets/


In [29]:
print("\nDiseases linked to stable genes:")
top_diseases


Diseases linked to stable genes:


,n_genes,avg_score,max_score
disease_name,,,
neurodegenerative disease,6,0.333517,0.481898
platelet count,5,0.224299,0.330540
metabolite measurement,4,0.310278,0.400547
high density lipoprotein cholesterol measurement,4,0.294278,0.403865
body height,4,0.259230,0.305826
gut microbiome measurement,4,0.255625,0.328101
lymphocyte count,4,0.242420,0.350884
pulse wave reflection index measurement,4,0.169751,0.251423
IGF-1 measurement,3,0.375908,0.431316


In [25]:
print("\nTherapeutic areas affected by stable genes:")
top_areas


Therapeutic areas affected by stable genes:


,n_genes,n_diseases,avg_score
therapeutic_areas,,,
measurement,16,163,0.263403
nervous system disease,14,15,0.279403
cancer or benign tumor,10,15,0.199125
phenotype,10,24,0.260112
reproductive system or breast disease,9,10,0.188460
gastrointestinal disease,8,15,0.234842
"genetic, familial or congenital disease",7,11,0.266234
integumentary system disease,5,4,0.236465
psychiatric disorder,5,6,0.230417


In [26]:
print("\n3 ultra-robust genes — top associations:")
print(ultra_robust_details.head(30).to_string(index=False))


3 ultra-robust genes — top associations:
        gene_id  selection_frequency                                              disease_name    disease_id                                                                                                          therapeutic_area_ids  association_score                                                                                               therapeutic_areas
ENSG00000047597                  1.0                       McLeod neuroacanthocytosis syndrome MONDO_0018945 {'list': [{'element': 'EFO_0000618'}, {'element': 'OTAR_0000018'}, {'element': 'MONDO_0002025'}, {'element': 'EFO_0000651'}]}           0.804145              [nervous system disease, genetic, familial or congenital disease, psychiatric disorder, phenotype]
ENSG00000047597                  1.0                                          genetic disorder   EFO_0000508                                                                                       {'list': [{'element': 'OTAR_000

NSC is like which genes, taken together, allow me to correctly classify a sample as treatment vs control?

1. A gene that changes a lot but doesn't help predict.
Suppose Gene X has expression: Control = [10, 2], Treatment = [3, 11, 1, 14, 0, 13, 2, 12, 4, 10]. The mean difference is small (~6), so log2FC flags it as "changed." But the spread within treatments is huge — knowing Gene X's value doesn't help you predict if a sample is treatment or control, because both classes have values across the whole range. NSC would not select this gene.
2. A gene that didn't change much but reliably predicts.
Gene Y expression: Control = [5.0, 5.1], Treatment = [4.0, 4.0, 4.1, 4.0, 4.1, 4.0, 4.0, 4.1, 4.0, 4.0]. The fold change is small (~20%), but the means are cleanly separated with low within-class variance. Knowing Gene Y's value perfectly predicts class. NSC would select it; log2FC might not even flag it as interesting.
3. Correlated genes — the most important difference.
If 10 genes all change identically because they're in the same biological pathway, log2FC flags all 10. NSC and especially LASSO might keep just 1 of them — because once you have one, the other 9 contain no additional predictive information.